In [0]:
# --- Install deps if needed ---
%pip install pymupdf pytesseract pillow pdf2image langchain langchain-experimental langchain-community databricks-vectorsearch

In [0]:
import os
import fitz
import pytesseract
from PIL import Image
from pyspark.sql import SparkSession

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import DatabricksEmbeddings
from databricks.vector_search.client import VectorSearchClient

# -----------------------------
# CONFIG
# -----------------------------
PDF_DIR = "/Volumes/workspace/default/studebakerrepairs"
DELTA_TABLE = "workspace.default.studebaker_semantic_chunks"
VECTOR_INDEX = "workspace.default.studebaker_semantic_index"
VECTOR_ENDPOINT = "vector_endpoint"

# -----------------------------
# SETUP
# -----------------------------
spark = SparkSession.builder.getOrCreate()

embedding_model = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)

chunker = SemanticChunker(embedding_model)

# -----------------------------
# PDF + OCR LOADER
# -----------------------------
def extract_text(pdf_path):

    doc = fitz.open(pdf_path)
    text = ""

    for page in doc:
        page_text = page.get_text()

        if page_text.strip():
            text += page_text
        else:
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text += pytesseract.image_to_string(img)

    return text


# -----------------------------
# PROCESS DIRECTORY
# -----------------------------
records = []

for dirpath, dirnames, filenames in os.walk(PDF_DIR):

    for file in filenames:

        if not file.lower().endswith(".pdf"):
            continue

        path = os.path.join(dirpath, file)

        print("Processing:", file)

        text = extract_text(path)

        chunks = chunker.split_text(text)

        embeddings = embedding_model.embed_documents(chunks)

        for i, chunk in enumerate(chunks):

            records.append({
                "chunk_id": f"{file}_{i}",
                "source": file,
                "text": chunk,
                "embedding": embeddings[i]
            })


# -----------------------------
# WRITE TO DELTA
# -----------------------------
df = spark.createDataFrame(records)

(df.write
 .mode("append")
 .saveAsTable(DELTA_TABLE))


# -----------------------------
# CREATE / SYNC VECTOR INDEX
# -----------------------------
client = VectorSearchClient()

try:
    client.create_delta_sync_index(
        endpoint_name=VECTOR_ENDPOINT,
        index_name=VECTOR_INDEX,
        source_table_name=DELTA_TABLE,
        primary_key="chunk_id",
        embedding_vector_column="embedding",
        embedding_dimension=len(records[0]["embedding"]),
        pipeline_type="TRIGGERED"
    )
except:
    print("Index already exists, syncing")

print("Pipeline complete.")

In [0]:
import os
import fitz
import pytesseract
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import DatabricksEmbeddings

# -----------------------------
# CONFIG
# -----------------------------
PDF_DIR = "/Volumes/workspace/default/studebakerrepairs"
DELTA_TABLE = "workspace.default.studebaker_semantic_chunks"

MAX_WORKERS = 8
EMBED_BATCH_SIZE = 64

embedding_model = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

# -----------------------------
# OCR
# -----------------------------
def extract_text(pdf_path):

    doc = fitz.open(pdf_path)
    text_parts = []

    for page in doc:

        page_text = page.get_text()

        if page_text.strip():
            text_parts.append(page_text)

        else:
            pix = page.get_pixmap()
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text_parts.append(pytesseract.image_to_string(img))

    return "\n".join(text_parts)


# -----------------------------
# PROCESS SINGLE FILE
# -----------------------------
def process_file(path):

    text = extract_text(path)

    chunks = splitter.split_text(text)

    return [
        {
            "chunk_id": f"{os.path.basename(path)}_{i}",
            "source": os.path.basename(path),
            "text": chunk
        }
        for i, chunk in enumerate(chunks)
    ]


# -----------------------------
# LOAD FILE LIST
# -----------------------------
files = []

for dirpath, _, filenames in os.walk(PDF_DIR):
    for f in filenames:
        if f.lower().endswith(".pdf"):
            files.append(os.path.join(dirpath, f))

print(f"Found {len(files)} PDFs")

# -----------------------------
# PARALLEL DOCUMENT PROCESSING
# -----------------------------
records = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [executor.submit(process_file, f) for f in files]

    for future in as_completed(futures):
        records.extend(future.result())


print(f"Generated {len(records)} chunks")

# -----------------------------
# BATCH EMBEDDINGS
# -----------------------------
texts = [r["text"] for r in records]

embeddings = []

for i in range(0, len(texts), EMBED_BATCH_SIZE):

    batch = texts[i:i+EMBED_BATCH_SIZE]

    embeddings.extend(
        embedding_model.embed_documents(batch)
    )

for i, vec in enumerate(embeddings):
    records[i]["embedding"] = vec

# -----------------------------
# WRITE TO DELTA TABLE
# -----------------------------
df = spark.createDataFrame(records)

(
    df.write
    .mode("append")
    .saveAsTable(DELTA_TABLE)
)

print("Pipeline complete.")